In [43]:
# Import required libraries
import os
from openai import OpenAI
import json
from collections import Counter
import time
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Set default model
MODEL = "gpt-4o-mini"

print("✅ Setup complete! OpenAI client initialized.")

✅ Setup complete! OpenAI client initialized.


In [44]:
def call_openai(prompt, system_message="You are a helpful assistant.", temperature=0.7, max_tokens=None, seed=None):
    """Helper function to call OpenAI API with specified parameters"""
    try:
        params = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature
        }
        
        if max_tokens:
            params["max_tokens"] = max_tokens
        if seed is not None:
            params["seed"] = seed
            
        response = client.chat.completions.create(**params)
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def display_response(title, response, params=None):
    """Display a response with formatting"""
    print(f"\n{'='*60}")
    print(f"📝 {title}")
    if params:
        print(f"Parameters: {params}")
    print(f"{'='*60}")
    print(response)
    print(f"{'='*60}\n")

def count_tokens_approx(text):
    """Approximate token count (roughly 4 characters per token)"""
    return len(text) // 4

print("✅ Helper functions loaded!")

✅ Helper functions loaded!


In [45]:
import time
import json
from collections import Counter

def run_multiple(prompt, n, temperature=0.7):
    """Send the same prompt n times and return all answers as a list."""
    responses = []
    for i in range(n):
        responses.append(call_openai(prompt, temperature=temperature))
        time.sleep(0.5)   # small pause to avoid rate limits
    return responses

def analyze(responses):
    """Measure how consistent a list of answers is."""
    cleaned = [r.strip().lower() for r in responses]
    counts = Counter(cleaned)
    top_answer, top_count = counts.most_common(1)[0]
    lengths = [len(r) for r in responses]
    return {
        "runs": len(responses),
        "unique": len(counts),
        "consistency_%": round(top_count / len(responses) * 100),
        "avg_len": round(sum(lengths) / len(lengths)),
        "min_len": min(lengths),
        "max_len": max(lengths),
    }

def is_valid_json(text):
    """True if the text can be read as JSON."""
    try:
        json.loads(text)
        return True
    except json.JSONDecodeError:
        return False

In [6]:
# Initial simple prompt for sentiment analysis
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""
 
# Test it once
result = call_openai(sentiment_prompt_v1)
print("Sentiment Analysis Result:")
print(result)

Sentiment Analysis Result:
This customer message can be classified as positive feedback or a positive review.


In [7]:
# Initial simple prompt for product description
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""
 
# Test it once
result = call_openai(product_prompt_v1)
print("Product Description Result:")
print(result)

Product Description Result:
**Product Description: Wireless Precision Mouse**

Elevate your computing experience with our Wireless Precision Mouse, designed for both functionality and style. Priced at just $29.99, this sleek and ergonomic mouse is perfect for home, office, or on-the-go use.

**Key Features:**

- **Seamless Wireless Connectivity:** Enjoy the freedom of movement with advanced 2.4GHz wireless technology. Say goodbye to tangled cords and enjoy a clutter-free workspace with a reliable connection up to 33 feet away.

- **Ergonomic Design:** Crafted with comfort in mind, our wireless mouse fits perfectly in your hand, reducing strain during long hours of use. Its contoured shape ensures a comfortable grip, ideal for both left and right-handed users.

- **High Precision Tracking:** Experience smooth and accurate navigation with our optical sensor that adapts to various surfaces. With adjustable DPI settings (800/1200/1600), you can customize your speed and precision for any ta

In [11]:
# Initial simple prompt for data extraction
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""
 
# Test it once
result = call_openai(extraction_prompt_v1)
print("Data Extraction Result:")
print(result)

Data Extraction Result:
Here’s the extracted information from the customer feedback:

- **Order Number**: 12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged


In [14]:
prompts_v1 = {
    "sentiment": sentiment_prompt_v1,
    "product": product_prompt_v1,
    "extraction": extraction_prompt_v1,
}

results = {}   # results["v1"][5]["sentiment"] -> list of 5 answers

def run_test(prompts, version, n):
    results.setdefault(version, {})[n] = {}
    for task, prompt in prompts.items():
        responses = run_multiple(prompt, n)
        results[version][n][task] = responses

        print(f"\n===== {task.upper()} | {version} | {n} runs =====")
        print(analyze(responses))

        if task == "sentiment":
            hits = sum("positive" in r.lower() for r in responses)
            print(f"Mentions 'positive': {hits}/{n}")
        if task == "extraction":
            ok = sum(is_valid_json(r) for r in responses)
            print(f"Valid JSON: {ok}/{n}")

        for i, r in enumerate(responses, 1):
            print(f"  Run {i}: {r[:150].replace(chr(10), ' ')}")

run_test(prompts_v1, "v1", 5)


===== SENTIMENT | v1 | 5 runs =====
{'runs': 5, 'unique': 4, 'consistency_%': 40, 'avg_len': 85, 'min_len': 82, 'max_len': 93}
Mentions 'positive': 5/5
  Run 1: This customer message can be classified as **positive feedback** or **satisfaction**.
  Run 2: The customer message can be classified as positive feedback or a customer compliment.
  Run 3: The customer message can be classified as **positive feedback** or **customer satisfaction**.
  Run 4: This customer message can be classified as positive feedback or a positive review.
  Run 5: This customer message can be classified as positive feedback or a positive review.

===== PRODUCT | v1 | 5 runs =====
{'runs': 5, 'unique': 5, 'consistency_%': 20, 'avg_len': 2025, 'min_len': 1872, 'max_len': 2295}
  Run 1: **Product Description: Wireless Precision Mouse - $29.99**  Elevate your computing experience with our Wireless Precision Mouse, designed for both wor
  Run 2: **Product Name:** SleekWave Wireless Mouse  **Price:** $29.99  **Prod

In [15]:
run_test(prompts_v1, "v1", 10)


===== SENTIMENT | v1 | 10 runs =====
{'runs': 10, 'unique': 7, 'consistency_%': 20, 'avg_len': 82, 'min_len': 64, 'max_len': 94}
Mentions 'positive': 10/10
  Run 1: The customer message can be classified as positive feedback or a positive review.
  Run 2: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
  Run 3: The customer message can be classified as "Positive Feedback" or "Satisfaction."
  Run 4: This customer message can be classified as **positive feedback** or **customer satisfaction**.
  Run 5: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
  Run 6: The customer message can be classified as **Positive Feedback**.
  Run 7: This customer message can be classified as positive feedback or a positive review.
  Run 8: This customer message can be classified as positive feedback or a positive review.
  Run 9: This customer message can be classified as **Positive Feedback**.
  Run 10: The cust

In [16]:
run_test(prompts_v1, "v1", 15)


===== SENTIMENT | v1 | 15 runs =====
{'runs': 15, 'unique': 9, 'consistency_%': 20, 'avg_len': 85, 'min_len': 64, 'max_len': 94}
Mentions 'positive': 15/15
  Run 1: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
  Run 2: This customer message can be classified as "Positive Feedback" or "Satisfaction."
  Run 3: The customer message can be classified as **Positive Feedback**.
  Run 4: The customer message can be classified as **Positive Feedback**.
  Run 5: This customer message can be classified as **positive feedback** or **customer satisfaction**.
  Run 6: This customer message can be classified as **positive feedback** or **customer satisfaction**.
  Run 7: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
  Run 8: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
  Run 9: The customer message can be classified as "Positive Feedback" or "Customer Satisfact

In [46]:
import pandas as pd

rows = []
for n, tasks in results["v1"].items():
    for task, responses in tasks.items():
        stats = analyze(responses)
        stats["task"] = task
        stats["runs"] = n
        if task == "sentiment":
            stats["extra_check"] = f"positive {sum('positive' in r.lower() for r in responses)}/{n}"
        elif task == "extraction":
            stats["extra_check"] = f"valid JSON {sum(is_valid_json(r) for r in responses)}/{n}"
        else:
            stats["extra_check"] = "-"
        rows.append(stats)

failure_table = pd.DataFrame(rows)[["task", "runs", "unique", "consistency_%", "avg_len", "min_len", "max_len", "extra_check"]]
failure_table.sort_values(["task", "runs"])

,task,runs,unique,consistency_%,avg_len,min_len,max_len,extra_check
2,extraction,5,4,40,174,158,180,valid JSON 0/5
5,extraction,10,5,40,182,180,194,valid JSON 0/10
8,extraction,15,9,33,178,158,180,valid JSON 0/15
1,product,5,5,20,2025,1872,2295,-
4,product,10,10,10,1875,1589,2048,-
7,product,15,15,7,1909,1666,2125,-
0,sentiment,5,4,40,85,82,93,positive 5/5
3,sentiment,10,7,20,82,64,94,positive 10/10
6,sentiment,15,9,20,85,64,94,positive 15/15


In [18]:
sentiment_prompt_v2 = """Classify the sentiment of the customer message below.

Allowed labels: positive, negative, neutral

Rules:
- Answer with exactly ONE label from the list.
- Lowercase, one word, no punctuation, no explanation.

Customer message: "I love this product! It's exactly what I needed."

Label:"""

product_prompt_v2 = """Write a product description for an online shop.

Product information:
- Product: wireless mouse
- Price: $29.99

Use exactly this structure:
Title: Wireless Mouse
Description: <2-3 sentences, 50-70 words>
Price: $29.99

Rules:
- Use ONLY the product information given. Do not invent brand names, specifications, measurements, colors or features.
- Friendly, clear, professional tone.
- Plain text only: no markdown, no bold, no bullet points."""

extraction_prompt_v2 = """Extract information from the customer feedback below.

Return ONLY a JSON object with exactly these keys:
- "item_id": the item number as a string, without "#"
- "order_date": the date exactly as written in the text
- "delivery": one or two words describing the delivery
- "packaging": one or two words describing the packaging
If a value is missing from the text, use null.

No explanation, no preamble, no markdown.

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

prompts_v2 = {
    "sentiment": sentiment_prompt_v2,
    "product": product_prompt_v2,
    "extraction": extraction_prompt_v2,
}

run_test(prompts_v2, "v2", 15)


===== SENTIMENT | v2 | 15 runs =====
{'runs': 15, 'unique': 1, 'consistency_%': 100, 'avg_len': 8, 'min_len': 8, 'max_len': 8}
Mentions 'positive': 15/15
  Run 1: positive
  Run 2: positive
  Run 3: positive
  Run 4: positive
  Run 5: positive
  Run 6: positive
  Run 7: positive
  Run 8: positive
  Run 9: positive
  Run 10: positive
  Run 11: positive
  Run 12: positive
  Run 13: positive
  Run 14: positive
  Run 15: positive

===== PRODUCT | v2 | 15 runs =====
{'runs': 15, 'unique': 15, 'consistency_%': 7, 'avg_len': 346, 'min_len': 306, 'max_len': 389}
  Run 1: Title: Wireless Mouse   Description: Experience the freedom of movement with our wireless mouse, designed for seamless navigation and comfort. Its sle
  Run 2: Title: Wireless Mouse   Description: Experience seamless navigation and effortless control with our wireless mouse. Designed for comfort and ease of u
  Run 3: Title: Wireless Mouse   Description: Experience the freedom of movement with our wireless mouse, designed for

In [21]:
import pandas as pd

def product_format_ok(text):
    """True if the product text follows the v2 structure and length."""
    if "Description:" not in text or "Price:" not in text:
        return False
    description = text.split("Description:")[1].split("Price:")[0]
    words = len(description.split())
    return (
        text.strip().startswith("Title:")
        and "$29.99" in text
        and "**" not in text
        and 50 <= words <= 70
    )

rows = []
for version, runs in results.items():
    for n, tasks in runs.items():
        for task, responses in tasks.items():
            stats = analyze(responses)
            stats["version"] = version
            stats["task"] = task
            stats["runs"] = n
            if task == "sentiment":
                stats["extra_check"] = f"positive {sum('positive' in r.lower() for r in responses)}/{n}"
            elif task == "extraction":
                stats["extra_check"] = f"valid JSON {sum(is_valid_json(r) for r in responses)}/{n}"
            else:
                stats["extra_check"] = f"format OK {sum(product_format_ok(r) for r in responses)}/{n}"
            rows.append(stats)

columns = ["task", "version", "runs", "unique", "consistency_%", "avg_len", "min_len", "max_len", "extra_check"]
comparison = pd.DataFrame(rows)[columns].sort_values(["task", "version", "runs"])
comparison

,task,version,runs,unique,consistency_%,avg_len,min_len,max_len,extra_check
2,extraction,v1,5,4,40,174,158,180,valid JSON 0/5
5,extraction,v1,10,5,40,182,180,194,valid JSON 0/10
8,extraction,v1,15,9,33,178,158,180,valid JSON 0/15
11,extraction,v2,15,1,100,102,102,102,valid JSON 15/15
1,product,v1,5,5,20,2025,1872,2295,format OK 0/5
4,product,v1,10,10,10,1875,1589,2048,format OK 0/10
7,product,v1,15,15,7,1909,1666,2125,format OK 0/15
10,product,v2,15,15,7,346,306,389,format OK 0/15
0,sentiment,v1,5,4,40,85,82,93,positive 5/5
3,sentiment,v1,10,7,20,82,64,94,positive 10/10


In [22]:
for r in results["v2"][15]["product"]:
    desc = r.split("Description:")[1].split("Price:")[0]
    print(
        len(desc.split()), "words |",
        "Title OK" if r.strip().startswith("Title:") else "NO TITLE", "|",
        "price OK" if "$29.99" in r else "NO PRICE", "|",
        "no markdown" if "**" not in r else "HAS **",
    )

46 words | Title OK | price OK | no markdown
48 words | Title OK | price OK | no markdown
45 words | Title OK | price OK | no markdown
41 words | Title OK | price OK | no markdown
45 words | Title OK | price OK | no markdown
44 words | Title OK | price OK | no markdown
44 words | Title OK | price OK | no markdown
46 words | Title OK | price OK | no markdown
45 words | Title OK | price OK | no markdown
40 words | Title OK | price OK | no markdown
42 words | Title OK | price OK | no markdown
43 words | Title OK | price OK | no markdown
44 words | Title OK | price OK | no markdown
48 words | Title OK | price OK | no markdown
41 words | Title OK | price OK | no markdown


In [30]:
for r in results["v3"][15]["product"]:
    desc = r.split("Description:")[1].split("Price:")[0]
    print(
        len(desc.split()), "words |",
        "Title OK" if r.strip().startswith("Title:") else "NO TITLE", "|",
        "price OK" if "$29.99" in r else "NO PRICE", "|",
        "no markdown" if "**" not in r else "HAS **",
    )

54 words | Title OK | price OK | no markdown
46 words | Title OK | price OK | no markdown
51 words | Title OK | price OK | no markdown
55 words | Title OK | price OK | no markdown
48 words | Title OK | price OK | no markdown
52 words | Title OK | price OK | no markdown
49 words | Title OK | price OK | no markdown
45 words | Title OK | price OK | no markdown
52 words | Title OK | price OK | no markdown
46 words | Title OK | price OK | no markdown
48 words | Title OK | price OK | no markdown
50 words | Title OK | price OK | no markdown
52 words | Title OK | price OK | no markdown
49 words | Title OK | price OK | no markdown
48 words | Title OK | price OK | no markdown


In [24]:
def get_answer(text):
    """Return only the final answer: the part after 'JSON:' if present."""
    if "JSON:" in text:
        text = text.split("JSON:")[-1]
    return text.strip()

def is_valid_json(text):
    try:
        json.loads(get_answer(text))
        return True
    except json.JSONDecodeError:
        return False

In [51]:
sentiment_prompt_v3 = """Classify the sentiment of a customer message.

Allowed labels: positive, negative, neutral
Answer with exactly ONE lowercase label and nothing else.

Examples:
Message: "The app crashes every time I open it. Very disappointed."
Label: negative

Message: "My order arrived on Tuesday."
Label: neutral

Message: "Great quality and super fast shipping, thank you!"
Label: positive

Message: "I love this product! It's exactly what I needed."
Label:"""

product_prompt_v3 = """Write a product description for an online shop.

Rules:
- Use ONLY the product information given. Do not invent brand names, specifications, measurements or colors.
- Exactly 3 sentences, about 60 words in total, like the example.
- Plain text only: no markdown, no bullet points.
- Use the same three lines as the example: Title, Description, Price.

Example
Product information: USB-C charger, $19.99
Title: USB-C Charger
Description: Keep your phone and other devices ready to go with this USB-C charger. Simply connect it to a USB-C cable and plug it into a power outlet to start charging. It is a practical everyday essential for your desk, your bedside table or your travel bag, and a smart choice for anyone who wants a dependable way to stay charged.
Price: $19.99

Now write the description for this product:
Product information: wireless mouse, $29.99"""

extraction_prompt_v3 = """Extract information from the customer feedback below.

First, think step by step:
1. Find the item number.
2. Find the order date.
3. Find what the customer says about delivery and about packaging.
4. Decide the overall sentiment: positive, negative or mixed (mixed = both positive and negative points).
5. Name the main issue in 2-3 lowercase words, or null if there is none.

Write your reasoning as a short numbered list.
Then write a line with only "JSON:" followed by a JSON object with exactly these keys:
"item_id" (string, without "#"), "order_date" (string, as written), "delivery" (1-2 words),
"packaging" (1-2 words), "overall_sentiment" (positive/negative/mixed), "main_issue" (string or null).
Use null for anything not in the text. No markdown after "JSON:".

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

prompts_v3 = {
    "sentiment": sentiment_prompt_v3,
    "product": product_prompt_v3,
    "extraction": extraction_prompt_v3,
}

run_test(prompts_v3, "v3", 15)


===== SENTIMENT | v3 | 15 runs =====
{'runs': 15, 'unique': 1, 'consistency_%': 100, 'avg_len': 8, 'min_len': 8, 'max_len': 8}
Mentions 'positive': 15/15
  Run 1: positive
  Run 2: positive
  Run 3: positive
  Run 4: positive
  Run 5: positive
  Run 6: positive
  Run 7: positive
  Run 8: positive
  Run 9: positive
  Run 10: positive
  Run 11: positive
  Run 12: positive
  Run 13: positive
  Run 14: positive
  Run 15: positive

===== PRODUCT | v3 | 15 runs =====
{'runs': 15, 'unique': 15, 'consistency_%': 7, 'avg_len': 372, 'min_len': 336, 'max_len': 400}
  Run 1: Title: Wireless Mouse   Description: Experience seamless navigation with this wireless mouse, designed for comfort and efficiency. Enjoy the freedom o
  Run 2: Title: Wireless Mouse   Description: Experience seamless navigation and enhanced productivity with this wireless mouse. Its ergonomic design ensures c
  Run 3: Title: Wireless Mouse   Description: Enhance your computing experience with this sleek wireless mouse. Design

In [52]:
from collections import Counter
answers = [get_answer(r) for r in results["v3"][15]["extraction"]]
for answer, count in Counter(answers).items():
    print(count, "×", answer)

9 × {"item_id":"12345","order_date":"March 15th","delivery":"fast","packaging":"damaged","overall_sentiment":"mixed","main_issue":"packaging"}
5 × {"item_id":"12345","order_date":"March 15th","delivery":"fast","packaging":"damaged","overall_sentiment":"mixed","main_issue":"damaged packaging"}
1 × {"item_id":"12345","order_date":"March 15th","delivery":"fast","packaging":"damaged","overall_sentiment":"mixed","main_issue":"packaging damage"}


In [32]:
def product_with_retry(prompt, max_tries=3):
    """Generate a product text; retry if it fails the format check."""
    for attempt in range(1, max_tries + 1):
        r = call_openai(prompt, temperature=0.7)
        if product_format_ok(r):
            return r, attempt
    return r, max_tries   # give back the last try if none passed

retry_outputs, attempts = [], []
for i in range(15):
    r, a = product_with_retry(product_prompt_v3)
    retry_outputs.append(r)
    attempts.append(a)

results["v3_retry"] = {15: {"product": retry_outputs}}
print("Format OK:", sum(product_format_ok(r) for r in retry_outputs), "/ 15")
print("Attempts needed:", attempts)
print("Total API calls:", sum(attempts))

Format OK: 11 / 15
Attempts needed: [2, 3, 2, 2, 3, 3, 1, 3, 3, 3, 2, 2, 3, 3, 1]
Total API calls: 36


In [35]:
def run_variations(make_prompt, cases, n=5, check=None):
    """Run each test case n times and measure consistency and correctness."""
    rows = []
    for case in cases:
        prompt = make_prompt(case)
        responses = [get_answer(r) for r in run_multiple(prompt, n)]
        stats = analyze(responses)
        stats["case"] = case["name"]
        stats["passed"] = f"{sum(check(r, case) for r in responses)}/{n}" if check else "-"
        stats["example_output"] = responses[0][:80].replace("\n", " ")
        rows.append(stats)
    return pd.DataFrame(rows)[["case", "unique", "consistency_%", "passed", "example_output"]]

In [53]:
SENT_ORIGINAL = "I love this product! It's exactly what I needed."

sentiment_cases = [
    {"name": "neutral fact",   "text": "The package arrived today.",                          "expected": "neutral"},
    {"name": "clear negative", "text": "Terrible service, I'm never ordering again.",         "expected": "negative"},
    {"name": "lukewarm",       "text": "It's okay, nothing special.",                         "expected": "neutral"},
    {"name": "mixed",          "text": "I wanted to love it, but it broke after two days.",   "expected": "negative"},
    {"name": "sarcasm",        "text": "Oh great, another delayed delivery. Just perfect.",   "expected": "negative"},
]

def make_sentiment(case):
    return sentiment_prompt_v3.replace(SENT_ORIGINAL, case["text"])

def sentiment_correct(r, case):
    return r.strip().lower() == case["expected"]

sentiment_variations = run_variations(make_sentiment, sentiment_cases, check=sentiment_correct)
sentiment_variations

,case,unique,consistency_%,passed,example_output
0,neutral fact,1,100,5/5,neutral
1,clear negative,1,100,5/5,negative
2,lukewarm,1,100,5/5,neutral
3,mixed,1,100,5/5,negative
4,sarcasm,1,100,5/5,negative


In [54]:
product_cases = [
    {"name": "headphones",   "info": "noise-cancelling headphones, $89.99",          "price": "$89.99"},
    {"name": "water bottle", "info": "stainless steel water bottle, 750 ml, $24.99", "price": "$24.99"},
    {"name": "yoga mat",     "info": "yoga mat, $34.99",                             "price": "$34.99"},
]

def make_product(case):
    return product_prompt_v3.replace(
        "Product information: wireless mouse, $29.99",
        f"Product information: {case['info']}",
    )

def product_ok(r, case):
    if "Description:" not in r or "Price:" not in r:
        return False
    words = len(r.split("Description:")[1].split("Price:")[0].split())
    return (r.strip().startswith("Title:") and case["price"] in r
            and "**" not in r and 50 <= words <= 70)

product_variations = run_variations(make_product, product_cases, check=product_ok)
product_variations

,case,unique,consistency_%,passed,example_output
0,headphones,5,20,3/5,Title: Noise-Cancelling Headphones Descripti...
1,water bottle,5,20,5/5,Title: Stainless Steel Water Bottle Descript...
2,yoga mat,5,20,2/5,Title: Yoga Mat Description: Elevate your pr...


In [55]:
EXTR_ORIGINAL = "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
KEYS = {"item_id", "order_date", "delivery", "packaging", "overall_sentiment", "main_issue"}

extraction_cases = [
    {"name": "complaint",   "text": "Order #98765 from June 2nd never arrived. I want a refund.",          "expected": "negative"},
    {"name": "no id, date", "text": "Love the blue jacket! It fits perfectly and came nicely wrapped.",   "expected": "positive"},
    {"name": "mixed",       "text": "Item 55501 arrived on time and well packed, but it is the wrong size.", "expected": "mixed"},
]

def make_extraction(case):
    return extraction_prompt_v3.replace(EXTR_ORIGINAL, case["text"])

def extraction_ok(r, case):
    try:
        data = json.loads(r)
    except json.JSONDecodeError:
        return False
    return set(data) == KEYS and data["overall_sentiment"] == case["expected"]

extraction_variations = run_variations(make_extraction, extraction_cases, check=extraction_ok)
extraction_variations

,case,unique,consistency_%,passed,example_output
0,complaint,2,80,5/5,"{""item_id"":""98765"",""order_date"":""June 2nd"",""de..."
1,"no id, date",3,60,5/5,"{""item_id"":null,""order_date"":null,""delivery"":""..."
2,mixed,1,100,5/5,"{""item_id"":""55501"",""order_date"":null,""delivery..."


In [70]:
print("Versions in results:", list(results.keys()))

rows = []
for version, runs in results.items():
    for n, tasks in runs.items():
        for task, responses in tasks.items():
            stats = analyze([get_answer(r) for r in responses])
            stats.update({"version": version, "task": task, "runs": n})
            if task == "sentiment":
                stats["extra_check"] = f"positive {sum('positive' in r.lower() for r in responses)}/{n}"
            elif task == "extraction":
                stats["extra_check"] = f"valid JSON {sum(is_valid_json(r) for r in responses)}/{n}"
            else:
                stats["extra_check"] = f"format OK {sum(product_format_ok(r) for r in responses)}/{n}"
            rows.append(stats)

columns = ["task", "version", "runs", "unique", "consistency_%", "avg_len", "min_len", "max_len", "extra_check"]
comparison = pd.DataFrame(rows)[columns].sort_values(["task", "version", "runs"])
final = comparison[comparison["runs"] == 15]
final

Versions in results: ['v1', 'v2', 'v3', 'v3_retry']


,task,version,runs,unique,consistency_%,avg_len,min_len,max_len,extra_check
8,extraction,v1,15,9,33,178,158,180,valid JSON 0/15
11,extraction,v2,15,1,100,102,102,102,valid JSON 15/15
14,extraction,v3,15,3,60,141,138,146,valid JSON 15/15
7,product,v1,15,15,7,1909,1666,2125,format OK 0/15
10,product,v2,15,15,7,346,306,389,format OK 0/15
13,product,v3,15,15,7,372,336,400,format OK 6/15
15,product,v3_retry,15,15,7,381,338,409,format OK 11/15
6,sentiment,v1,15,9,20,85,64,94,positive 15/15
9,sentiment,v2,15,1,100,8,8,8,positive 15/15
12,sentiment,v3,15,1,100,8,8,8,positive 15/15


In [68]:
print(repr(get_answer(results["v3"][15]["extraction"][0])))

'{"item_id":"12345","order_date":"March 15th","delivery":"fast","packaging":"damaged","overall_sentiment":"mixed","main_issue":"packaging"}'


In [69]:
def get_answer(text):
    """Return only the final answer: the JSON object if there is one."""
    if "JSON:" in text:
        text = text.split("JSON:")[-1]
    text = text.strip()
    if "{" in text and "}" in text:
        text = text[text.find("{") : text.rfind("}") + 1]
    return text

def is_valid_json(text):
    try:
        json.loads(get_answer(text))
        return True
    except json.JSONDecodeError:
        return False

sample = results["v3"][15]["extraction"][0]
print("End of answer:", repr(get_answer(sample)[-40:]))
print("Valid JSON?", is_valid_json(sample))

End of answer: 'iment":"mixed","main_issue":"packaging"}'
Valid JSON? True


## Final Evaluation – v1 vs v3 (15 runs, temperature 0.7)

| Task | v1 | v3 | Technique that helped most |
|---|---|---|---|
| Sentiment | 20% consistent, 9 unique, invented labels | 100%, 1 unique | Allowed labels + one-word rule (v2); few-shot kept it robust |
| Extraction | 0/15 valid JSON | 15/15 valid JSON, correct "mixed" sentiment | Exact JSON keys (v2); CoT for judgment fields (v3) |
| Product | ~1,900 chars, hallucinated brands/specs, format 0/15 | ~370 chars, no invented specs, format 6/15 (11/15 with retry) | Matched few-shot example; code validation + retry |

**Notes**
- v3 extraction consistency is 60% only because the free-text `main_issue` varies ("packaging" vs "damaged packaging"); all other fields are identical in 15/15 runs.
- One "failure" was in my own checking code (an outdated `is_valid_json`), not the model: always verify the measuring tool first.
- CoT made the output chattier (reasoning before the answer), so the JSON had to be cut out with a parser.

## Part 5 – Task Variations (v3 prompts, 5 runs per case)

- **Sentiment:** 25/25 correct, 100% consistent, incl. neutral, lukewarm, mixed and sarcasm. The few-shot examples generalise well.
- **Extraction:** 15/15 correct keys and sentiment; missing values returned as null (no invented IDs or dates).
- **Product:** success depends on the number of facts: water bottle (3 facts) 5/5, headphones (2) 3/5, yoga mat (1) 2/5. This confirms the conflict between "do not invent" and a fixed length when input information is minimal.

**What needs adjustment:** a fixed list of values for `main_issue`; the product length target should scale with the amount of product information, or the client should provide more product details.